In [ ]:
import numpy as np
import os
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
feature_name = '1024feature'
non_mace_file_dir = fr'V:\dunwei\MACE\dataset\ECG_Founder_result\sr500_0.5_50_10s_ECG_signal\sr500_0.5_50_10s_ECG_signal_{feature_name}\non_mace/'
mace_file_dir = fr'V:\dunwei\MACE\dataset\ECG_Founder_result\sr500_0.5_50_10s_ECG_signal\sr500_0.5_50_10s_ECG_signal_{feature_name}\mace/'
save_dir = fr'V:\dunwei\MACE\dataset\ECG_Founder_result\sr500_0.5_50_10s_ECG_signal\sr500_0.5_50_10s_ECG_signal_{feature_name}\correlation_coefficient/'

In [ ]:
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f'Create directory: {save_dir}')
else:
    print(f'Directory already exists: {save_dir}')

In [ ]:
non_mace_signals_list = []
non_mace_labels_list = []
for i, non_mace_filename in enumerate(tqdm(os.listdir(non_mace_file_dir))):
    try:
        if non_mace_filename.endswith('.npy'):
            non_mace_data = np.load(os.path.join(non_mace_file_dir, non_mace_filename))
            non_mace_signal = non_mace_data[:, 1:]
            non_mace_label = non_mace_data[:, 0]
            non_mace_signals_list.append(non_mace_signal)
            non_mace_labels_list.append(non_mace_label)

    except Exception as e:
        print(f"Error loading {non_mace_filename}: {e}")
non_mace_signals_arr = np.vstack(non_mace_signals_list)
non_mace_labels_arr = np.hstack(non_mace_labels_list)
print(non_mace_signals_arr.shape)
print(non_mace_labels_arr.shape)

In [ ]:
mace_signals_list = []
mace_labels_list = []
for i, mace_filename in enumerate(tqdm(os.listdir(mace_file_dir), desc='Loading mace files')):
    try:
        if mace_filename.endswith('.npy'):
            mace_data = np.load(os.path.join(mace_file_dir, mace_filename))
            mace_signal = mace_data[:, 1:]
            mace_label = mace_data[:, 0]
            mace_signals_list.append(mace_signal)
            mace_labels_list.append(mace_label)

    except Exception as e:
        print(f"Error loading {mace_filename}: {e}")

mace_signals_arr = np.vstack(mace_signals_list)
mace_labels_arr = np.hstack(mace_labels_list)
print(mace_signals_arr.shape)
print(mace_labels_arr.shape)

In [ ]:
all_signals_arr = np.vstack((non_mace_signals_arr, mace_signals_arr))
all_labels_arr = np.hstack((non_mace_labels_arr, mace_labels_arr))
print(all_signals_arr.shape)
print(all_labels_arr.shape)

In [ ]:
thresholds = [0.9, 0.8, 0.7, 0.6, 0.5]
unique_pairs_list = []
# correlation coefficient matrix
correlation_matrix = np.corrcoef(all_signals_arr, rowvar=False) # correlation_matrix output (N_feature, N_feature)
np.fill_diagonal(correlation_matrix, 0)  # Fill diagonal with 0 to ignore self-correlation
for threshold in thresholds:
    highly_correlated_pairs = np.where(np.abs(correlation_matrix) > threshold)
    unique_pairs = [(r, c) for r, c in zip(*highly_correlated_pairs) if r < c]
    unique_pairs_list.append(unique_pairs)

print(len(unique_pairs_list))

In [ ]:
feature_label_correlations = []
for i in range(all_signals_arr.shape[1]):
    corr, _ = pearsonr(all_signals_arr[:, i], all_labels_arr) # correlation output (N_feature, )，calculate correlation between each feature and the label
    feature_label_correlations.append(abs(corr))
feature_label_correlations = np.array(feature_label_correlations)
print(feature_label_correlations.shape)

In [ ]:
features_to_remove_list = []
for i, unique_pair in enumerate(unique_pairs_list):
    features_to_remove = set()
    for r, c in unique_pair:
        corr_r_to_label = feature_label_correlations[r]
        corr_c_to_label = feature_label_correlations[c]
        if corr_r_to_label < corr_c_to_label:
            features_to_remove.add(r)
        else:
            features_to_remove.add(c)
    features_to_remove_list.append(features_to_remove)

# print(features_to_remove_list)

In [ ]:
result_thresholds_dict = {'cc_threshold': [], 'num_features_removed': []}
for i in range(len(features_to_remove_list)):
    result_thresholds_dict['cc_threshold'].append(thresholds[i])
    result_thresholds_dict['num_features_removed'].append(len(features_to_remove_list[i]))
result_thresholds_df = pd.DataFrame(result_thresholds_dict)
print(result_thresholds_df)
# result_thresholds_df.to_csv(os.path.join(save_dif, 'two_groups_correlation_coefficient_results.csv'), index=False)

In [ ]:
# for i in range(len(features_to_remove_list)):
#     features_to_keep = np.array(list(set(np.arange(signals_arr.shape[1])) - features_to_remove_list[i]))
#     print(features_to_keep.shape)